In [ ]:
import struct
import pandas as pd

entries = []

pos = 0

while pos < len(central_data):

    signature = central_data[pos:pos+4]

    if signature != b"PK\x01\x02":
        break

    filename_length = struct.unpack(
        "<H", central_data[pos+28:pos+30]
    )[0]

    extra_length = struct.unpack(
        "<H", central_data[pos+30:pos+32]
    )[0]

    comment_length = struct.unpack(
        "<H", central_data[pos+32:pos+34]
    )[0]

    compressed_size = struct.unpack(
        "<I", central_data[pos+20:pos+24]
    )[0]

    uncompressed_size = struct.unpack(
        "<I", central_data[pos+24:pos+28]
    )[0]

    local_header_offset = struct.unpack(
        "<I", central_data[pos+42:pos+46]
    )[0]

    filename_start = pos + 46
    filename_end = filename_start + filename_length

    filename = central_data[
        filename_start:filename_end
    ].decode("utf-8")

    entries.append({
        "filename": filename,
        "compressed_size": compressed_size,
        "uncompressed_size": uncompressed_size,
        "local_header_offset": local_header_offset
    })

    pos += (
        46
        + filename_length
        + extra_length
        + comment_length
    )

zip_index = pd.DataFrame(entries)

print("Total ZIP entries:", len(zip_index))
display(zip_index.head())

In [ ]:
def get_patient_entry(patient_id):
    """
    Find a RibFrac CT image inside the remote ZIP archive.
    """

    filename = f"Part1/{patient_id}-image.nii.gz"

    matches = zip_index[
        zip_index["filename"] == filename
    ]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"{filename} was not found in the RibFrac Part 1 image archive."
        )

    return matches.iloc[0].to_dict()

In [ ]:
import requests
import io
import zlib
import os

IMAGE_URL = (
    "https://zenodo.org/records/3893508/files/"
    "ribfrac-train-images-1.zip"
)

def download_ribfrac_image(patient_id, output_dir="/kaggle/working/ribfrac_images"):

    os.makedirs(output_dir, exist_ok=True)

    entry = get_patient_entry(patient_id)

    local_header_offset = entry["local_header_offset"]
    compressed_size = entry["compressed_size"]

    # ----------------------------------------------------
    # Read local ZIP header
    # ----------------------------------------------------

    header_end = local_header_offset + 29

    response = requests.get(
        IMAGE_URL,
        headers={
            "Range": f"bytes={local_header_offset}-{header_end}"
        },
        timeout=120
    )

    header = response.content

    if len(header) != 30:
        raise RuntimeError(
            f"Could not retrieve ZIP header for {patient_id}."
        )

    (
        signature,
        version_needed,
        flags,
        compression_method,
        mod_time,
        mod_date,
        crc32,
        header_compressed_size,
        header_uncompressed_size,
        filename_length,
        extra_length
    ) = struct.unpack(
        "<4s5H3I2H",
        header
    )

    if signature != b"PK\x03\x04":
        raise RuntimeError(
            f"Invalid ZIP local header for {patient_id}."
        )

    # ----------------------------------------------------
    # Calculate compressed data location
    # ----------------------------------------------------

    data_start = (
        local_header_offset
        + 30
        + filename_length
        + extra_length
    )

    data_end = data_start + compressed_size - 1

    # ----------------------------------------------------
    # Download compressed member in chunks
    # ----------------------------------------------------

    CHUNK_SIZE = 1024 * 1024  # 1 MB

    chunks = []

    current_start = data_start

    print(f"Downloading {patient_id}...")
    print(f"Compressed size: {compressed_size / (1024**2):.2f} MB")

    while current_start <= data_end:

        current_end = min(
            current_start + CHUNK_SIZE - 1,
            data_end
        )

        response = requests.get(
            IMAGE_URL,
            headers={
                "Range": f"bytes={current_start}-{current_end}"
            },
            timeout=120
        )

        if response.status_code != 206:
            raise RuntimeError(
                f"Range request failed: "
                f"{response.status_code}"
            )

        expected = current_end - current_start + 1

        if len(response.content) != expected:
            raise RuntimeError(
                f"Expected {expected} bytes, "
                f"received {len(response.content)} bytes."
            )

        chunks.append(response.content)

        current_start = current_end + 1

        downloaded = current_start - data_start

        print(
            f"\rDownloaded: "
            f"{downloaded / (1024**2):.2f} / "
            f"{compressed_size / (1024**2):.2f} MB",
            end=""
        )

    print("\nDownload complete.")

    compressed_data = b"".join(chunks)

    # ----------------------------------------------------
    # Decompress ZIP DEFLATE data
    # ----------------------------------------------------

    print("Decompressing ZIP member...")

    decompressed_data = zlib.decompress(
        compressed_data,
        -15
    )

    print(
        f"Decompressed size: "
        f"{len(decompressed_data) / (1024**2):.2f} MB"
    )

    # ----------------------------------------------------
    # Save .nii.gz
    # ----------------------------------------------------

    output_path = os.path.join(
        output_dir,
        f"{patient_id}-image.nii.gz"
    )

    with open(output_path, "wb") as f:
        f.write(decompressed_data)

    print("Saved:", output_path)

    return output_path

In [ ]:
ct_path = download_ribfrac_image("RibFrac128")

In [ ]:
import nibabel as nib

ct_img = nib.load(ct_path)

print("CT loaded successfully")
print("-----------------------------")
print("Shape:", ct_img.shape)
print("Data type:", ct_img.get_data_dtype())
print("Voxel spacing:", ct_img.header.get_zooms()[:3])